In [1]:
"""
Combined Stress Detection: WESAD + SWELL (HR-Only, Apple Watch Compatible)
==========================================================================
Uses HR data from both datasets to build a more robust stress detector.
 
WESAD stress = Trier Social Stress Test (public speaking, mental arithmetic)
SWELL stress = Office work under time pressure + interruptions
 
Combining both teaches the model multiple "flavors" of stress.
 
Usage:
  1. Place these files in the same directory:
     - wesad-chest-combined-classification-hrv.csv  (from Kaggle)
     - combined-swell-classification-hrv-dataset.csv (from Kaggle)
  2. Run this script
  3. Use predict_stress() with your Apple Watch HR readings
"""
 
import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
import joblib
import warnings
warnings.filterwarnings('ignore')
 
 
# ============================================================================
# FEATURE ENGINEERING (Mandal et al. approach)
# ============================================================================
 
def compute_features(hr_chunk, global_stats):
    """
    Compare a window of HR readings to the person's baseline.
    Stress = deviation from YOUR normal pattern.
    """
    hr = np.array(hr_chunk, dtype=float)
    
    local_mean = np.mean(hr)
    local_median = np.median(hr)
    local_max = np.max(hr)
    local_min = np.min(hr)
    local_std = np.std(hr)
    local_range = local_max - local_min
    
    g = global_stats
    
    # Ratios: how does this moment compare to my normal?
    mean_ratio = local_mean / g['mean'] if g['mean'] != 0 else 1.0
    median_ratio = local_median / g['median'] if g['median'] != 0 else 1.0
    max_ratio = local_max / g['max'] if g['max'] != 0 else 1.0
    min_ratio = local_min / g['min'] if g['min'] != 0 else 1.0
    std_ratio = local_std / g['std'] if g['std'] != 0 else 1.0
    
    # Squared differences: how far from baseline?
    mean_diff = np.sqrt((local_mean - g['mean'])**2)
    median_diff = np.sqrt((local_median - g['median'])**2)
    max_diff = np.sqrt((local_max - g['max'])**2)
    min_diff = np.sqrt((local_min - g['min'])**2)
    
    # Trend within chunk
    if len(hr) >= 4:
        half = len(hr) // 2
        trend = np.mean(hr[half:]) - np.mean(hr[:half])
    else:
        trend = 0.0
    
    # Coefficient of variation
    cv = local_std / local_mean if local_mean != 0 else 0.0
    
    return {
        'mean_ratio': mean_ratio,
        'median_ratio': median_ratio,
        'max_ratio': max_ratio,
        'min_ratio': min_ratio,
        'std_ratio': std_ratio,
        'mean_diff': mean_diff,
        'median_diff': median_diff,
        'max_diff': max_diff,
        'min_diff': min_diff,
        'local_range': local_range,
        'local_std': local_std,
        'trend': trend,
        'cv': cv,
        'local_mean': local_mean,
    }
 
 
def build_chunks(hr_series, label_series, subject_series, chunk_size=20, stride=5):
    """Build sliding-window chunks from HR time series, grouped by subject."""
    
    # Compute per-subject global baselines
    df_temp = pd.DataFrame({'HR': hr_series, 'subject': subject_series})
    baselines = {}
    for subj, grp in df_temp.groupby('subject'):
        hrs = grp['HR'].values
        baselines[subj] = {
            'mean': np.mean(hrs),
            'median': np.median(hrs),
            'max': np.max(hrs),
            'min': np.min(hrs),
            'std': np.std(hrs),
        }
    
    all_features = []
    all_labels = []
    all_subjects = []
    
    # Group by subject to maintain temporal order
    df_full = pd.DataFrame({
        'HR': hr_series, 
        'label': label_series, 
        'subject': subject_series
    })
    
    for subj, grp in df_full.groupby('subject'):
        hrs = grp['HR'].values
        labels = grp['label'].values
        bl = baselines[subj]
        
        for i in range(0, len(hrs) - chunk_size + 1, stride):
            chunk_hr = hrs[i:i + chunk_size]
            chunk_labels = labels[i:i + chunk_size]
            
            # Majority vote for label
            chunk_label = 1 if np.mean(chunk_labels) > 0.5 else 0
            
            features = compute_features(chunk_hr, bl)
            all_features.append(features)
            all_labels.append(chunk_label)
            all_subjects.append(subj)
    
    return pd.DataFrame(all_features), np.array(all_labels), np.array(all_subjects), baselines
 
 
# ============================================================================
# LOAD DATA
# ============================================================================
 
print("=" * 70)
print("LOADING DATASETS")
print("=" * 70)
 
all_hr = []
all_labels = []
all_subjects = []
datasets_loaded = []
 
# --- Load WESAD ---
try:
    wesad = pd.read_csv("wesad-chest-combined-classification-hrv.csv")
    
    if 'condition label' in wesad.columns:
        wesad.rename(columns={'condition label': 'Condition Label'}, inplace=True)
    
    # Map: baseline(0) + amusement(1) = no stress, stress(2) = stress
    wesad_stress = wesad['Condition Label'].map({0: 0, 1: 0, 2: 1})
    
    if 'subject id' in wesad.columns:
        wesad_subj = 'wesad_' + wesad['subject id'].astype(str)
    elif 'subject_id' in wesad.columns:
        wesad_subj = 'wesad_' + wesad['subject_id'].astype(str)
    
    # Only rows with valid HR
    mask = wesad['HR'].notna()
    all_hr.extend(wesad.loc[mask, 'HR'].values.tolist())
    all_labels.extend(wesad_stress[mask].values.tolist())
    all_subjects.extend(wesad_subj[mask].values.tolist())
    
    n_wesad = mask.sum()
    datasets_loaded.append(f"WESAD: {n_wesad} HR readings, "
                          f"{wesad_subj[mask].nunique()} subjects")
    print(f"  Loaded WESAD: {n_wesad} readings")
except FileNotFoundError:
    print("  WESAD CSV not found — skipping")
 
# --- Load SWELL ---
try:
    swell = pd.read_csv("combined-swell-classification-hrv-dataset.csv")
    
    # Map: no_stress(0) = 0, interruption(1) + time_pressure(2) = stress
    swell_stress = swell['Condition Label'].map({0: 0, 1: 1, 2: 1})
    swell_subj = 'swell_' + swell['subject_id'].astype(str)
    
    mask = swell['HR'].notna()
    all_hr.extend(swell.loc[mask, 'HR'].values.tolist())
    all_labels.extend(swell_stress[mask].values.tolist())
    all_subjects.extend(swell_subj[mask].values.tolist())
    
    n_swell = mask.sum()
    datasets_loaded.append(f"SWELL: {n_swell} HR readings, "
                          f"{swell_subj[mask].nunique()} subjects")
    print(f"  Loaded SWELL: {n_swell} readings")
except FileNotFoundError:
    print("  SWELL CSV not found — skipping")
 
# --- Load from .numbers file (fallback/additional) ---
try:
    swell_numbers = pd.read_csv("swell_hr_from_numbers.csv")
    swell_n_subj = 'swell_num_' + swell_numbers['subject'].astype(str)
    
    all_hr.extend(swell_numbers['HR'].values.tolist())
    all_labels.extend(swell_numbers['stress_label'].values.tolist())
    all_subjects.extend(swell_n_subj.values.tolist())
    
    datasets_loaded.append(f"SWELL (.numbers): {len(swell_numbers)} HR readings, "
                          f"{swell_numbers['subject'].nunique()} subjects")
    print(f"  Loaded SWELL from .numbers: {len(swell_numbers)} readings")
except FileNotFoundError:
    print("  swell_hr_from_numbers.csv not found — skipping")
 
if len(all_hr) == 0:
    print("\n  NO DATA FOUND. Creating synthetic demo data...")
    np.random.seed(42)
    for subj_id in range(1, 21):
        # Rest readings
        n_rest = np.random.randint(40, 80)
        base_hr = np.random.uniform(60, 85)
        rest_hr = np.random.normal(base_hr, 3, n_rest)
        all_hr.extend(rest_hr.tolist())
        all_labels.extend([0] * n_rest)
        all_subjects.extend([f'synth_{subj_id}'] * n_rest)
        
        # Stress readings (elevated + more variable)
        n_stress = np.random.randint(30, 60)
        stress_hr = np.random.normal(base_hr + np.random.uniform(3, 15), 5, n_stress)
        all_hr.extend(stress_hr.tolist())
        all_labels.extend([1] * n_stress)
        all_subjects.extend([f'synth_{subj_id}'] * n_stress)
    
    datasets_loaded.append("Synthetic: 20 subjects (demo)")
 
all_hr = np.array(all_hr, dtype=float)
all_labels = np.array(all_labels, dtype=int)
all_subjects = np.array(all_subjects)
 
print(f"\n  COMBINED: {len(all_hr)} total HR readings")
print(f"  Subjects: {len(np.unique(all_subjects))}")
print(f"  Labels: no_stress={np.sum(all_labels==0)}, stress={np.sum(all_labels==1)}")
for d in datasets_loaded:
    print(f"    - {d}")
 
 
# ============================================================================
# BUILD FEATURES & TRAIN
# ============================================================================
 
print("\n" + "=" * 70)
print("BUILDING FEATURES & TRAINING")
print("=" * 70)
 
CHUNK_SIZE = 15  # Slightly smaller chunks work better with sparser data
STRIDE = 3
 
features_df, labels_arr, subjects_arr, all_baselines = build_chunks(
    all_hr, all_labels, all_subjects, 
    chunk_size=CHUNK_SIZE, stride=STRIDE
)
 
print(f"  Chunks: {len(features_df)}")
print(f"  Labels: no_stress={np.sum(labels_arr==0)}, stress={np.sum(labels_arr==1)}")
 
# Split by subject
unique_subjs = np.unique(subjects_arr)
np.random.seed(42)
np.random.shuffle(unique_subjs)
split = int(len(unique_subjs) * 0.8)
train_subjs = set(unique_subjs[:split])
test_subjs = set(unique_subjs[split:])
 
train_mask = np.isin(subjects_arr, list(train_subjs))
test_mask = np.isin(subjects_arr, list(test_subjs))
 
X_train, y_train = features_df[train_mask].values, labels_arr[train_mask]
X_test, y_test = features_df[test_mask].values, labels_arr[test_mask]
 
print(f"  Train: {len(X_train)} chunks ({len(train_subjs)} subjects)")
print(f"  Test:  {len(X_test)} chunks ({len(test_subjs)} subjects)")
 
# Cross-validation
model = HistGradientBoostingClassifier(
    max_iter=300, max_depth=8, learning_rate=0.1,
    min_samples_leaf=10, random_state=42,
)
 
kfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = []
for ti, vi in kfold.split(X_train, y_train):
    model.fit(X_train[ti], y_train[ti])
    cv_scores.append(accuracy_score(y_train[vi], model.predict(X_train[vi])))
print(f"\n  CV Accuracy: {np.mean(cv_scores):.3f} +/- {np.std(cv_scores):.3f}")
 
# Final train
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
 
print(f"  Test Accuracy: {accuracy_score(y_test, y_pred):.3f}")
print(f"\n{classification_report(y_test, y_pred, target_names=['No Stress', 'Stress'])}")
print(f"Confusion Matrix:\n{confusion_matrix(y_test, y_pred)}")
 
# Save
FEATURE_NAMES = list(features_df.columns)
joblib.dump(model, 'stress_model_combined.pkl')
joblib.dump(FEATURE_NAMES, 'feature_names_combined.pkl')
print("\nSaved: stress_model_combined.pkl, feature_names_combined.pkl")
 
 

LOADING DATASETS
  Loaded WESAD: 135650 readings
  Loaded SWELL: 391638 readings
  swell_hr_from_numbers.csv not found — skipping

  COMBINED: 527288 total HR readings
  Subjects: 37
  Labels: no_stress=307104, stress=220184
    - WESAD: 135650 HR readings, 15 subjects
    - SWELL: 391638 HR readings, 22 subjects

BUILDING FEATURES & TRAINING
  Chunks: 175604
  Labels: no_stress=122424, stress=53180
  Train: 138094 chunks (29 subjects)
  Test:  37510 chunks (8 subjects)

  CV Accuracy: 0.881 +/- 0.002
  Test Accuracy: 0.808

              precision    recall  f1-score   support

   No Stress       0.79      0.98      0.88     26179
      Stress       0.90      0.41      0.56     11331

    accuracy                           0.81     37510
   macro avg       0.85      0.69      0.72     37510
weighted avg       0.83      0.81      0.78     37510

Confusion Matrix:
[[25687   492]
 [ 6725  4606]]

Saved: stress_model_combined.pkl, feature_names_combined.pkl


In [ ]:

# ============================================================================
# INFERENCE FUNCTION
# ============================================================================
 
def predict_stress(hr_readings, global_baseline):
    """
    Predict stress from Apple Watch HR readings.
    
    hr_readings: list of recent HR (BPM) values (at least 15)
    global_baseline: dict with keys: mean, median, max, min, std
                     (computed from hours of your normal HR data)
    """
    mdl = joblib.load('stress_model_combined.pkl')
    fnames = joblib.load('feature_names_combined.pkl')
    
    chunk = hr_readings[-15:]  # Use last 15 readings
    feats = compute_features(chunk, global_baseline)
    vec = np.array([[feats[f] for f in fnames]])
    
    pred = mdl.predict(vec)[0]
    probs = mdl.predict_proba(vec)[0]
    
    return {
        'prediction': 'ANXIOUS' if pred == 1 else 'NOT ANXIOUS',
        'stress_probability': float(probs[1]),
        'confidence': float(max(probs)),
    }
 
 # ============================================================================
# REAL TEST CASES (from your .numbers file)
# ============================================================================

print("\n" + "=" * 70)
print("REAL TEST CASES (from your hrv_stress_labels.numbers)")
print("=" * 70)

TEST_CASES = [
    {"name": "p12 - rest", "expected": "NOT ANXIOUS", "person": "p12",
     "hr": [73, 70, 72, 76, 74, 72, 78, 82, 79, 82, 82, 85, 84, 83, 79]},
    {"name": "p21 - rest (calm)", "expected": "NOT ANXIOUS", "person": "p21",
     "hr": [68, 69, 70, 72, 67, 76, 74, 73, 76, 76, 75, 72, 73, 75, 76]},
    {"name": "p16 - rest (high HR)", "expected": "NOT ANXIOUS", "person": "p16",
     "hr": [90, 88, 87, 86, 87, 88, 90, 91, 90, 86, 89, 82, 90, 86, 87]},
    {"name": "p2 - no stress (office)", "expected": "NOT ANXIOUS", "person": "p2",
     "hr": [77, 75, 74, 75, 75, 77, 76, 77, 79, 76, 77, 76, 78, 77, 79]},
    {"name": "p16 - interruption", "expected": "ANXIOUS", "person": "p16",
     "hr": [81, 82, 80, 80, 83, 83, 82, 84, 81, 80, 78, 80, 76, 78, 73]},
    {"name": "p17 - interruption", "expected": "ANXIOUS", "person": "p17",
     "hr": [68, 69, 65, 69, 66, 67, 68, 71, 67, 66, 67, 68, 69, 66, 65]},
    {"name": "p19 - interruption", "expected": "ANXIOUS", "person": "p19",
     "hr": [83, 77, 82, 79, 77, 81, 79, 77, 79, 83, 81, 77, 77, 79, 81]},
    {"name": "p3 - interruption (spike)", "expected": "ANXIOUS", "person": "p3",
     "hr": [89, 95, 97, 100, 95, 93, 95, 94, 93, 95, 97, 93, 94, 95, 93]},
    {"name": "p7 - time pressure", "expected": "ANXIOUS", "person": "p7",
     "hr": [66, 66, 67, 71, 68, 67, 68, 72, 72, 72, 72, 70, 75, 72, 69]},
    # === HR.csv (Empatica E4 wrist sensor, 1Hz downsampled to 1/5sec) ===
    # Full session baseline: mean=75.1, std=10.7, range=55-135
    {"name": "E4 - deep rest (t=71min, HR~56)",
     "expected": "NOT ANXIOUS", "person": "e4",
     "hr": [57, 56, 55, 55, 55, 55, 55, 55, 55, 55, 56, 57, 57, 58, 59]},
    {"name": "E4 - steady calm (t=5min, HR~76)",
     "expected": "NOT ANXIOUS", "person": "e4",
     "hr": [75, 75, 75, 76, 76, 76, 76, 76, 76, 76, 76, 76, 76, 76, 76]},
    {"name": "E4 - steady calm (t=46min, HR~70)",
     "expected": "NOT ANXIOUS", "person": "e4",
     "hr": [71, 71, 71, 71, 71, 71, 71, 70, 70, 70, 70, 70, 70, 70, 70]},
    {"name": "E4 - peak stress (t=82min, HR~123, spike to 135!)",
     "expected": "ANXIOUS", "person": "e4",
     "hr": [126, 128, 132, 133, 135, 133, 130, 125, 121, 117, 114, 112, 111, 112, 111]},
    {"name": "E4 - high arousal (t=84min, HR~111)",
     "expected": "ANXIOUS", "person": "e4",
     "hr": [110, 106, 105, 105, 106, 106, 107, 109, 109, 111, 112, 115, 117, 119, 122]},
    {"name": "E4 - ramping up (t=36min, HR 87->123)",
     "expected": "ANXIOUS", "person": "e4",
     "hr": [87, 91, 96, 101, 106, 111, 115, 119, 120, 122, 123, 121, 120, 116, 110]},
    {"name": "E4 - rapid spike (t=81min, HR 70->123)",
     "expected": "ANXIOUS", "person": "e4",
     "hr": [70, 70, 70, 72, 73, 74, 78, 82, 87, 92, 98, 105, 111, 118, 123]},
    {"name": "E4 - coming down (t=85min, HR 126->74)",
     "expected": "ANXIOUS", "person": "e4",
     "hr": [124, 126, 126, 122, 118, 114, 109, 103, 97, 90, 87, 82, 79, 75, 74]},
]

# Person baselines from your .numbers file + HR.csv
BASELINES = {
    "e4":  {"mean": 75.1, "median": 74.5, "max": 134.8, "min": 54.6, "std": 10.7},
    "p12": {"mean": 79.9, "median": 82.0, "max": 88.0, "min": 70.0, "std": 4.7},
    "p21": {"mean": 72.8, "median": 73.0, "max": 79.0, "min": 64.0, "std": 2.3},
    "p16": {"mean": 82.1, "median": 86.0, "max": 92.0, "min": 73.0, "std": 3.1},
    "p2":  {"mean": 74.3, "median": 76.0, "max": 82.0, "min": 64.0, "std": 3.2},
    "p17": {"mean": 72.4, "median": 74.0, "max": 94.0, "min": 59.0, "std": 5.2},
    "p19": {"mean": 78.6, "median": 79.0, "max": 83.0, "min": 71.0, "std": 2.4},
    "p3":  {"mean": 92.2, "median": 95.0, "max": 107.0, "min": 72.0, "std": 3.4},
    "p7":  {"mean": 72.0, "median": 74.0, "max": 82.0, "min": 57.0, "std": 2.9},
}

correct, total = 0, 0
print()
for tc in TEST_CASES:
    bl = BASELINES[tc["person"]]
    try:
        result = predict_stress(tc["hr"], bl)
    except Exception as e:
        print(f"  [ERR ] {tc['name']} — {e}")
        continue

    ok = (tc["expected"] == result["prediction"])
    if ok: correct += 1
    total += 1

    tag = " OK " if ok else "MISS"
    hr_arr = np.array(tc["hr"])
    print(f"  [{tag}] {tc['name']}")
    print(f"         Expected: {tc['expected']:14s} | Got: {result['prediction']:14s} "
          f"| Stress prob: {result['stress_probability']:.1%} "
          f"| HR mean={hr_arr.mean():.0f}")

if total > 0:
    print(f"\n  Test case accuracy: {correct}/{total} ({correct/total*100:.0f}%)")
else:
    print("\n  No test cases completed.")


REAL TEST CASES (from your hrv_stress_labels.numbers)

  [ OK ] p12 - rest
         Expected: NOT STRESSED   | Got: NOT STRESSED   | Stress prob: 1.2% | HR mean=78
  [ OK ] p21 - rest (calm)
         Expected: NOT STRESSED   | Got: NOT STRESSED   | Stress prob: 2.8% | HR mean=73
  [ OK ] p16 - rest (high HR)
         Expected: NOT STRESSED   | Got: NOT STRESSED   | Stress prob: 8.8% | HR mean=88
  [ OK ] p2 - no stress (office)
         Expected: NOT STRESSED   | Got: NOT STRESSED   | Stress prob: 6.9% | HR mean=77
  [MISS] p16 - interruption
         Expected: STRESSED       | Got: NOT STRESSED   | Stress prob: 35.0% | HR mean=80
  [ OK ] p17 - interruption
         Expected: STRESSED       | Got: STRESSED       | Stress prob: 98.1% | HR mean=67
  [MISS] p19 - interruption
         Expected: STRESSED       | Got: NOT STRESSED   | Stress prob: 3.1% | HR mean=79
  [MISS] p3 - interruption (spike)
         Expected: STRESSED       | Got: NOT STRESSED   | Stress prob: 1.2% | HR mean=95
 